# GT Mask Generation & XAI Grid Rebuild

Generates **coloured ground-truth masks** from `gt_annotations_test.json` for the
100 original CheXlocalize test images, then rebuilds the 6-panel XAI grid by
**replacing the 'Original' panel** with the GT mask.

Mask colours match the XAI notebook's 14-colour palette exactly (indexed by
CheXpert label order) so colours are consistent across GT masks and XAI heatmaps.

---
### Input
- `gt_annotations_test.json` — radiologist polygon annotations
- `chexpert_output/test_samples/manifest.csv` — 100-image CheXlocalize manifest
- `xai_output/chexlocalize/*_xai.png` — saved XAI grid PNGs (2×3 panels)

### Output
- `gt_masks/` — individual coloured GT mask PNGs (one per image)
- `xai_output_with_gt/` — rebuilt 6-panel grids with GT mask as panel 1

### Panel layout (rebuilt grid)
```
[GT Mask] [LayerCAM    ] [FPN-LayerCAM]
[ScoreCAM] [FPN-ScoreCAM] [LIME        ]
```

## 0. Imports

In [ ]:
import json
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

## 1. Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
GT_ANNOTATIONS_PATH  = Path('./gt_annotations_test.json')
CHEX_MANIFEST        = Path('./chexpert_output/test_samples/manifest.csv')
CHEX_IMAGE_DIR       = Path('./chexpert_output/test_samples')  # saved 100 JPGs
XAI_GRID_DIR         = Path('./xai_output/chexlocalize')       # saved *_xai.png

GT_MASK_DIR          = Path('./gt_masks')                      # coloured masks output
REBUILT_GRID_DIR     = Path('./xai_output_with_gt')            # rebuilt grids output
GT_MASK_DIR.mkdir(parents=True, exist_ok=True)
REBUILT_GRID_DIR.mkdir(parents=True, exist_ok=True)

# ── CheXpert label space (model + XAI notebook colour index) ───────────────────
CHEXPERT_LABELS: List[str] = [
    'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia',
    'Atelectasis', 'Pneumothorax', 'Effusion',
    'Pleural Other', 'Fracture', 'Support Devices', 'No Finding',
]

# ── XAI notebook colour palette (14 colours, indexed by CHEXPERT_LABELS) ──────
# Must match LABEL_COLOURS in the XAI notebook exactly
LABEL_COLOURS: List[str] = [
    '#FF3333', '#FF8C00', '#FFD700', '#7FFF00', '#00CED1',
    '#1E90FF', '#DA70D6', '#FF69B4', '#00FA9A', '#FF6347',
    '#40E0D0', '#BA55D3', '#F0E68C', '#87CEEB',
]

# ── GT label → CheXpert label name mapping ─────────────────────────────────────
# CheXlocalize uses 'Airspace Opacity' and 'Pleural Effusion'
# CheXpert model uses 'Lung Opacity' and 'Effusion'
GT_TO_CHEXPERT: Dict[str, str] = {
    'Enlarged Cardiomediastinum': 'Enlarged Cardiomediastinum',
    'Cardiomegaly'              : 'Cardiomegaly',
    'Airspace Opacity'          : 'Lung Opacity',     # renamed in CheXpert
    'Lung Lesion'               : 'Lung Lesion',
    'Edema'                     : 'Edema',
    'Consolidation'             : 'Consolidation',
    'Atelectasis'               : 'Atelectasis',
    'Pneumothorax'              : 'Pneumothorax',
    'Pleural Effusion'          : 'Effusion',         # renamed in CheXpert
    'Support Devices'           : 'Support Devices',
}

# ── Display settings ───────────────────────────────────────────────────────────
IMG_SIZE          = 224    # XAI heatmaps were computed at 224×224
MASK_ALPHA        = 0.60   # opacity of coloured mask over original image
GRID_DPI          = 130    # must match XAI notebook dpi=130
GRID_FIGSIZE      = (13, 9)# must match XAI notebook figsize=(13, 9)

# Pre-compute colour lookup: GT label name → RGB tuple
def hex_to_rgb(h: str) -> Tuple[int, int, int]:
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

GT_LABEL_COLOURS: Dict[str, Tuple[int,int,int]] = {}
for gt_lbl, chex_lbl in GT_TO_CHEXPERT.items():
    if chex_lbl in CHEXPERT_LABELS:
        idx = CHEXPERT_LABELS.index(chex_lbl)
        GT_LABEL_COLOURS[gt_lbl] = hex_to_rgb(LABEL_COLOURS[idx])

print('GT label → colour mapping:')
for lbl, rgb in GT_LABEL_COLOURS.items():
    idx = CHEXPERT_LABELS.index(GT_TO_CHEXPERT[lbl])
    print(f'  {lbl:<30s}  idx={idx:2d}  {LABEL_COLOURS[idx]}  RGB={rgb}')

assert GT_ANNOTATIONS_PATH.exists(), f'Not found: {GT_ANNOTATIONS_PATH}'
assert CHEX_MANIFEST.exists(),       f'Not found: {CHEX_MANIFEST}'
print('\nConfiguration ready.')

## 2. GT Annotation Loader

In [ ]:
with open(GT_ANNOTATIONS_PATH) as f:
    GT_ANNOTATIONS: Dict = json.load(f)
print(f'Loaded annotations for {len(GT_ANNOTATIONS)} images.')


def filename_to_gt_key(filename: str) -> str:
    """
    Converts saved manifest filename to GT annotation lookup key.
    'patient64541_study1_view1_frontal.jpg'
    → 'patient64541_study1_view1_frontal'
    """
    return Path(filename).stem


def rasterise_label(
    img_key   : str,
    gt_label  : str,
    out_h     : int,
    out_w     : int,
) -> Optional[np.ndarray]:
    """
    Rasterises all polygons for one (image, label) into a binary mask.
    Returns uint8 mask at (out_h, out_w), or None if no annotation exists.
    Polygons are in [x, y] format at the original img_size resolution;
    the mask is resized to (out_h, out_w) after rasterisation.
    """
    if img_key not in GT_ANNOTATIONS:
        return None
    entry = GT_ANNOTATIONS[img_key]
    if gt_label not in entry:
        return None

    H_orig, W_orig = entry['img_size']
    mask = np.zeros((H_orig, W_orig), dtype=np.uint8)

    for polygon in entry[gt_label]:
        pts = np.array(polygon, dtype=np.float32)
        if len(pts) >= 3:
            cv2.fillPoly(mask, [pts.astype(np.int32).reshape(-1, 1, 2)], 1)

    if mask.sum() == 0:
        return None

    # Resize to output resolution with nearest-neighbour to keep binary values
    resized = cv2.resize(mask, (out_w, out_h), interpolation=cv2.INTER_NEAREST)
    return resized.astype(np.uint8)

## 3. Coloured GT Mask Generator

In [ ]:
def make_coloured_gt_mask(
    img_key     : str,
    img_rgb     : np.ndarray,
    out_h       : int = IMG_SIZE,
    out_w       : int = IMG_SIZE,
    alpha       : float = MASK_ALPHA,
) -> Tuple[np.ndarray, List[str]]:
    """
    Builds a coloured GT mask overlaid on the original image.

    Strategy:
      - Each GT label is rasterised to a binary mask.
      - Pixels are assigned the colour of the label with the largest
        annotated area at that location (winner-takes-all per pixel,
        consistent with the XAI composite heatmap rendering).
      - The coloured mask is alpha-blended over the original image
        so anatomy remains visible.
      - Labels without any annotation for this image are skipped.

    Returns:
      composite   : (H, W, 3) uint8 — coloured overlay
      found_labels: list of GT label names that had non-empty masks
    """
    colour_layer = np.zeros((out_h, out_w, 3), dtype=np.float32)
    area_layer   = np.zeros((out_h, out_w),    dtype=np.float32)  # tracks winner
    found_labels : List[str] = []

    for gt_label, rgb in GT_LABEL_COLOURS.items():
        mask = rasterise_label(img_key, gt_label, out_h, out_w)
        if mask is None:
            continue
        found_labels.append(gt_label)

        # Winner-takes-all: only update pixels where this mask's area
        # equals 1 and no previous label has claimed that pixel,
        # OR this label's mask is present (all annotated pixels count equally).
        # Since each pixel can only belong to one annotated region in most
        # cases, we update wherever this mask fires and current area is 0.
        dominant = (mask == 1) & (area_layer == 0)
        # Also overwrite if this is a new label claiming unclaimed pixels
        colour_layer[mask == 1] = np.array(rgb, dtype=np.float32)
        area_layer  [mask == 1] = 1.0  # mark as claimed

    if len(found_labels) == 0:
        # No annotations: return original image with a grey tint
        return img_rgb.copy(), []

    # Alpha blend: coloured region opaque at `alpha`, transparent elsewhere
    alpha_map  = (area_layer * alpha)[..., np.newaxis]   # (H, W, 1)
    canvas     = img_rgb.astype(np.float32)
    composite  = canvas * (1 - alpha_map) + colour_layer * alpha_map
    return np.clip(composite, 0, 255).astype(np.uint8), found_labels


def build_gt_legend_patches(
    found_labels: List[str],
    gt_labels_in_image: Optional[List[str]] = None,
) -> List[mpatches.Patch]:
    """
    Legend patches for GT mask panel.
    Shows only labels that have a polygon annotation for this image.
    gt_labels_in_image: the list of GT labels from manifest (ground truth);
    used to mark ✓ (annotation present) only — all found_labels are correct
    by definition since they come from the GT file.
    """
    patches = []
    for lbl in found_labels:
        colour = LABEL_COLOURS[CHEXPERT_LABELS.index(GT_TO_CHEXPERT[lbl])]
        # Display name: use CheXpert label name for consistency with XAI legend
        display_name = GT_TO_CHEXPERT[lbl]
        patches.append(mpatches.Patch(
            color=colour,
            label=f'\u2713 {display_name} (GT)',
        ))
    return patches

## 4. XAI Grid Panel Extractor

In [ ]:
def extract_xai_panels(
    grid_path : Path,
) -> Dict[int, np.ndarray]:
    """
    Extracts the 5 XAI panels (indices 1–5) from a saved 2×3 XAI grid PNG.

    The XAI notebook saved grids with:
      figsize=(13, 9), dpi=130  →  canvas = 1690 × 1170 px
      tight_layout(rect=[0, 0.08, 1, 1])  → bottom 8% is legend space
      2 rows × 3 cols, equal subplot sizes

    Panel layout in the grid:
      [0=Original  ] [1=LayerCAM    ] [2=FPN-LayerCAM]
      [3=ScoreCAM  ] [4=FPN-ScoreCAM] [5=LIME        ]

    We only need panels 1–5 (the XAI heatmaps).

    Returns dict {panel_index: np.ndarray (H, W, 3) uint8}

    NOTE: Rather than hard-coding pixel offsets, we use a robust approach:
      1. Read the full grid image.
      2. Determine the subplot content area by detecting where the white
         title text region begins (top margin) and the legend begins (bottom).
      3. Split the content area into a 2×3 grid of equal cells.
    This handles minor DPI/tight_layout variations gracefully.
    """
    grid_img = np.array(Image.open(grid_path).convert('RGB'))
    H, W     = grid_img.shape[:2]

    # ── Detect subplot region boundaries ─────────────────────────────────────
    # Matplotlib tight_layout with rect=[0, 0.08, 1, 1]:
    #   top margin    ≈ 4% of H (suptitle)
    #   bottom margin ≈ 8% of H (legend)
    # Use conservative fixed fractions that work for figsize=(13,9), dpi=130:
    top_frac    = 0.05   # skip suptitle
    bottom_frac = 0.10   # skip legend
    left_frac   = 0.04   # skip left spine labels
    right_frac  = 0.02   # skip right margin

    y0 = int(H * top_frac)
    y1 = int(H * (1.0 - bottom_frac))
    x0 = int(W * left_frac)
    x1 = int(W * (1.0 - right_frac))

    content_h = y1 - y0
    content_w = x1 - x0
    row_h     = content_h // 2
    col_w     = content_w // 3

    panels: Dict[int, np.ndarray] = {}
    for row in range(2):
        for col in range(3):
            panel_idx = row * 3 + col
            py0 = y0 + row * row_h
            py1 = py0 + row_h
            px0 = x0 + col * col_w
            px1 = px0 + col_w

            # Trim axis label area (≈6% of cell from each inner edge)
            trim_y = int(row_h * 0.10)
            trim_x = int(col_w * 0.06)
            panel = grid_img[
                py0 + trim_y : py1 - trim_y // 2,
                px0 + trim_x : px1 - trim_x // 2,
            ]
            panels[panel_idx] = panel

    return panels  # {0: original, 1: LayerCAM, 2: FPN-LayerCAM,
                   #  3: ScoreCAM, 4: FPN-ScoreCAM, 5: LIME}

## 5. Grid Rebuilder

In [ ]:
XAI_PANEL_TITLES = [
    'LayerCAM',
    'FPN-LayerCAM',
    'ScoreCAM',
    'FPN-ScoreCAM',
    'LIME (SLIC)',
]


def rebuild_grid(
    gt_overlay     : np.ndarray,
    gt_patches     : List[mpatches.Patch],
    xai_panels     : Dict[int, np.ndarray],
    xai_patches    : List[mpatches.Patch],
    image_name     : str,
    output_path    : Path,
) -> None:
    """
    Builds the new 2x3 figure with TWO visually separate legends:

      Row 1: [GT Mask] [LayerCAM] [FPN-LayerCAM]
      Row 2: [ScoreCAM] [FPN-ScoreCAM] [LIME]
      Row 3: [Legend 1: GT annotations] | [Legend 2: Detected labels]

    Legend 1 (left third): ground-truth annotations with tick markers.
    Legend 2 (right two-thirds): model-detected labels with tick/cross markers.
    A thin vertical divider and distinct bold blue titles separate them.
    """
    panels_data = [
        (gt_overlay,        'Ground-truth mask'),
        (xai_panels.get(1), 'LayerCAM'),
        (xai_panels.get(2), 'FPN-LayerCAM'),
        (xai_panels.get(3), 'ScoreCAM'),
        (xai_panels.get(4), 'FPN-ScoreCAM'),
        (xai_panels.get(5), 'LIME (SLIC)'),
    ]

    # 3-row layout: 2 image rows + 1 legend row
    fig = plt.figure(figsize=GRID_FIGSIZE)
    gs_outer = fig.add_gridspec(
        3, 1,
        height_ratios=[1, 1, 0.22],
        hspace=0.18,
    )

    # Image rows: 2 x 3
    gs_imgs = gs_outer[:2, 0].subgridspec(2, 3, hspace=0.10, wspace=0.04)
    axes    = [fig.add_subplot(gs_imgs[r, c]) for r in range(2) for c in range(3)]

    for ax, (panel_img, title) in zip(axes, panels_data):
        if panel_img is not None:
            ax.imshow(panel_img)
        else:
            ax.set_facecolor('#EEEEEE')
            ax.text(0.5, 0.5, 'Not available',
                    ha='center', va='center', transform=ax.transAxes,
                    fontsize=9, color='#999999')
        ax.set_title(title, fontsize=10, pad=5)
        ax.axis('off')

    # Legend row: left (GT) | right (XAI)
    gs_leg    = gs_outer[2, 0].subgridspec(1, 2, wspace=0.0, width_ratios=[1, 2])
    ax_leg_gt = fig.add_subplot(gs_leg[0, 0])
    ax_leg_xai= fig.add_subplot(gs_leg[0, 1])
    for ax in [ax_leg_gt, ax_leg_xai]:
        ax.set_axis_off()

    # Legend 1 — Ground-truth annotations
    if gt_patches:
        n_col_gt = min(len(gt_patches), 2)
        leg_gt = ax_leg_gt.legend(
            handles        = gt_patches,
            loc            = 'upper left',
            bbox_to_anchor = (0.0, 1.0),
            ncol           = n_col_gt,
            fontsize       = 7.5,
            frameon        = True,
            framealpha     = 0.85,
            edgecolor      = '#888888',
            title          = '\u25ae Ground-truth annotations',
            title_fontsize = 8,
            labelspacing   = 0.35,
            handlelength   = 1.2,
            handleheight   = 0.9,
            borderpad      = 0.5,
        )
        leg_gt.get_title().set_fontweight('bold')
        leg_gt.get_title().set_color('#1F4E79')

    # Vertical divider between legends
    # Use ax.plot in axes coordinates via transAxes to avoid axvline's
    # transform restriction — draws a line at x=0 of ax_leg_xai.
    ax_leg_xai.plot(
        [0.0, 0.0], [0.05, 0.95],
        color='#AAAAAA', linewidth=0.8,
        transform=ax_leg_xai.transAxes,
        clip_on=False,
    )

    # Legend 2 — Detected labels
    if xai_patches:
        n_col_xai = min(len(xai_patches), 4)
        leg_xai = ax_leg_xai.legend(
            handles        = xai_patches,
            loc            = 'upper left',
            bbox_to_anchor = (0.02, 1.0),
            ncol           = n_col_xai,
            fontsize       = 7.5,
            frameon        = True,
            framealpha     = 0.85,
            edgecolor      = '#888888',
            title          = '\u25ae Detected labels  (\u2713 correct  \u2717 false positive)',
            title_fontsize = 8,
            labelspacing   = 0.35,
            handlelength   = 1.2,
            handleheight   = 0.9,
            borderpad      = 0.5,
        )
        leg_xai.get_title().set_fontweight('bold')
        leg_xai.get_title().set_color('#1F4E79')

    fig.suptitle(image_name, fontsize=9, y=1.005)
    plt.savefig(output_path, dpi=GRID_DPI, bbox_inches='tight')
    plt.close(fig)


## 6. Standalone GT Mask Saver

In [ ]:
def save_standalone_gt_mask(
    img_key     : str,
    img_rgb     : np.ndarray,
    found_labels: List[str],
    output_path : Path,
) -> None:
    """
    Saves a standalone PNG showing:
      Left : original image
      Right: coloured GT mask overlay
    with a colour legend below.
    """
    gt_overlay, _ = make_coloured_gt_mask(img_key, img_rgb)

    # Also make a mask-only version (black background, pure colours)
    H, W = img_rgb.shape[:2]
    mask_only = np.zeros((H, W, 3), dtype=np.uint8)
    for gt_label in found_labels:
        mask = rasterise_label(img_key, gt_label, H, W)
        if mask is not None:
            rgb = GT_LABEL_COLOURS[gt_label]
            mask_only[mask == 1] = rgb

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_rgb);      axes[0].set_title('Original',      fontsize=10); axes[0].axis('off')
    axes[1].imshow(mask_only);    axes[1].set_title('GT mask (pure)', fontsize=10); axes[1].axis('off')
    axes[2].imshow(gt_overlay);   axes[2].set_title('GT mask overlay',fontsize=10); axes[2].axis('off')

    patches = []
    for lbl in found_labels:
        colour = LABEL_COLOURS[CHEXPERT_LABELS.index(GT_TO_CHEXPERT[lbl])]
        patches.append(mpatches.Patch(color=colour, label=GT_TO_CHEXPERT[lbl]))

    if patches:
        fig.legend(handles=patches, loc='lower center', ncol=min(len(patches), 5),
                   fontsize=9, title='GT labels', title_fontsize=9,
                   bbox_to_anchor=(0.5, -0.02))

    fig.suptitle(Path(output_path).stem, fontsize=9)
    plt.tight_layout(rect=[0, 0.08, 1, 1])
    plt.savefig(output_path, dpi=120, bbox_inches='tight')
    plt.close(fig)

## 7. Build XAI Detected-Label Legend from Saved Grid

The original XAI grid legend is embedded in the PNG and cannot be extracted as
patches. We reconstruct it from the model's saved predictions in the manifest.

In [ ]:
def build_xai_legend_from_manifest(
    row           : pd.Series,
    threshold     : float = 0.5,
) -> List[mpatches.Patch]:
    """
    Reconstructs XAI detected-label legend patches from manifest row.

    The manifest stores binary GT labels per CheXpert label name.
    We use these as ground-truth reference (✓/✗ markers).
    Since we don't have per-image model probabilities in the manifest,
    we mark all GT-positive labels as ✓ (present in the image)
    and note in the title that colours match the XAI heatmap overlay.

    For a fully accurate reconstruction of the XAI legend with probabilities,
    use the results CSV (results_chexlocalize.csv) to look up each image's
    detected classes and their probabilities — see the commented extension below.
    """
    patches = []
    for lbl in CHEXPERT_LABELS:
        # Check if this label has a GT annotation for this image
        gt_val = float(row.get(lbl, 0.0))
        if gt_val > 0:
            idx    = CHEXPERT_LABELS.index(lbl)
            colour = LABEL_COLOURS[idx]
            patches.append(mpatches.Patch(
                color=colour,
                label=f'\u2713 {lbl}',
            ))
    return patches


def build_xai_legend_from_results(
    filename      : str,
    results_df    : pd.DataFrame,
    gt_row        : pd.Series,
    threshold     : float = 0.5,
) -> List[mpatches.Patch]:
    """
    Preferred: reconstructs the exact XAI legend with probabilities
    and ✓/✗ markers from the evaluation results CSV.

    results_df: loaded from results_chexlocalize.csv (or _413 version).
    Filters to this filename, takes unique detected labels and their
    mean probability across XAI methods (all methods predicted the same
    classes since they share the same model threshold pass).
    """
    img_rows = results_df[results_df['filename'] == filename]
    if len(img_rows) == 0:
        return build_xai_legend_from_manifest(gt_row)

    # Get unique (label, prob) pairs — use any single method (all share same probs)
    method_rows = img_rows[img_rows['method'] == 'LayerCAM']
    if len(method_rows) == 0:
        method_rows = img_rows.groupby('label').first().reset_index()

    patches = []
    for _, r in method_rows.iterrows():
        lbl    = r['label']
        prob   = r['prob']
        gt_val = float(gt_row.get(lbl, 0.0))
        symbol = '\u2713' if gt_val > 0 else '\u2717'
        idx    = CHEXPERT_LABELS.index(lbl) if lbl in CHEXPERT_LABELS else -1
        if idx < 0:
            continue
        colour = LABEL_COLOURS[idx]
        patches.append(mpatches.Patch(
            color=colour,
            label=f'{symbol} {lbl}  {prob:.2f}',
        ))
    return patches

## 8. Main Pipeline

In [ ]:
# ── Clear previous outputs so grids are regenerated with the new layout ──────
# Run this block once after updating rebuild_grid; comment out for future runs.
import shutil
for _dir in [REBUILT_GRID_DIR]:
    for _f in _dir.glob('*_xai_with_gt.png'):
        _f.unlink()
print(f'Cleared {len(list(REBUILT_GRID_DIR.glob("*.png")))} old rebuilt grids.')
print('Re-generating all grids with two-legend layout...')

# ── Load manifest and optional results CSV ────────────────────────────────────
manifest = pd.read_csv(CHEX_MANIFEST)
print(f'Manifest: {len(manifest)} images')

# Try to load results CSV for accurate XAI legend reconstruction
results_df = None
for results_path in [
    Path('./xai_evaluation_full/results_chexlocalize_413.csv'),
    Path('./xai_evaluation/results_chexlocalize.csv'),
]:
    if results_path.exists():
        results_df = pd.read_csv(results_path)
        print(f'Results CSV loaded: {results_path} ({len(results_df)} rows)')
        break

if results_df is None:
    print('Results CSV not found — XAI legend will use GT labels only.')

# ── Processing loop ───────────────────────────────────────────────────────────
skipped_no_xai  = []
skipped_no_img  = []
skipped_no_ann  = []
processed       = 0

for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc='Rebuilding grids'):
    filename = row['filename']
    img_key  = filename_to_gt_key(filename)

    # ── Locate source files ───────────────────────────────────────────────────
    img_path  = CHEX_IMAGE_DIR / filename
    stem      = Path(filename).stem
    grid_path = XAI_GRID_DIR / f'{stem}_xai.png'

    if not img_path.exists():
        skipped_no_img.append(filename); continue
    if not grid_path.exists():
        skipped_no_xai.append(filename); continue

    # ── Load original image ───────────────────────────────────────────────────
    img_rgb = np.array(
        Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE)),
        dtype=np.uint8
    )

    # ── Generate coloured GT mask overlay ────────────────────────────────────
    gt_overlay, found_labels = make_coloured_gt_mask(img_key, img_rgb)

    if not found_labels:
        skipped_no_ann.append(filename)
        # Still rebuild the grid — panel 0 will show the image with a
        # 'No GT annotation' label rather than silently skipping

    # ── Save standalone GT mask PNG ───────────────────────────────────────────
    save_standalone_gt_mask(
        img_key      = img_key,
        img_rgb      = img_rgb,
        found_labels = found_labels,
        output_path  = GT_MASK_DIR / f'{stem}_gt_mask.png',
    )

    # ── Extract XAI panels from saved grid ────────────────────────────────────
    xai_panels = extract_xai_panels(grid_path)

    # ── Build legend patches ──────────────────────────────────────────────────
    gt_patches  = build_gt_legend_patches(found_labels)
    xai_patches = (
        build_xai_legend_from_results(filename, results_df, row)
        if results_df is not None
        else build_xai_legend_from_manifest(row)
    )

    # ── Rebuild 6-panel grid ──────────────────────────────────────────────────
    out_path = REBUILT_GRID_DIR / f'{stem}_xai_with_gt.png'
    if out_path.exists():
        processed += 1
        continue   # resume-friendly

    rebuild_grid(
        gt_overlay  = gt_overlay,
        gt_patches  = gt_patches,
        xai_panels  = xai_panels,
        xai_patches = xai_patches,
        image_name  = filename,
        output_path = out_path,
    )
    processed += 1

print(f'\nDone.')
print(f'  Processed     : {processed}')
print(f'  No GT annot.  : {len(skipped_no_ann)} (grid still rebuilt, panel 0 = image)')
print(f'  No XAI grid   : {len(skipped_no_xai)}')
print(f'  No image file : {len(skipped_no_img)}')
if skipped_no_xai:
    print(f'  Missing grids : {skipped_no_xai[:5]}')

## 9. Spot-Check — Display Sample Rebuilt Grid

In [ ]:
def display_rebuilt_sample(idx: int = 0) -> None:
    grids = sorted(REBUILT_GRID_DIR.glob('*_xai_with_gt.png'))
    if not grids:
        print('No rebuilt grids found yet.')
        return
    chosen = grids[idx % len(grids)]
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.imshow(np.array(Image.open(chosen)))
    ax.axis('off')
    ax.set_title(chosen.name, fontsize=9)
    plt.tight_layout()
    plt.show()
    print(f'Grid: {chosen}')


def display_gt_mask_sample(idx: int = 0) -> None:
    masks = sorted(GT_MASK_DIR.glob('*_gt_mask.png'))
    if not masks:
        print('No GT masks found yet.')
        return
    chosen = masks[idx % len(masks)]
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.imshow(np.array(Image.open(chosen)))
    ax.axis('off')
    ax.set_title(chosen.name, fontsize=9)
    plt.tight_layout()
    plt.show()


print('Rebuilt grids:')
display_rebuilt_sample(idx=0)

print('Standalone GT masks:')
display_gt_mask_sample(idx=0)

## 10. Output Summary

In [ ]:
rebuilt_grids = list(REBUILT_GRID_DIR.glob('*_xai_with_gt.png'))
gt_masks      = list(GT_MASK_DIR.glob('*_gt_mask.png'))

print('=' * 58)
print('  GT Mask Generation & Grid Rebuild — Summary')
print('=' * 58)
print(f'  Standalone GT masks : {len(gt_masks):>4}  →  {GT_MASK_DIR.resolve()}')
print(f'  Rebuilt XAI grids   : {len(rebuilt_grids):>4}  →  {REBUILT_GRID_DIR.resolve()}')
print()
print('  New panel layout (per rebuilt grid):')
print('    Row 1: [GT Mask overlay] [LayerCAM    ] [FPN-LayerCAM]')
print('    Row 2: [ScoreCAM        ] [FPN-ScoreCAM] [LIME        ]')
print()
print('  GT mask colours match XAI heatmap palette:')
for gt_lbl, rgb in GT_LABEL_COLOURS.items():
    chex_lbl = GT_TO_CHEXPERT[gt_lbl]
    idx = CHEXPERT_LABELS.index(chex_lbl)
    print(f'    {chex_lbl:<30s} #{hex(rgb[0]).upper()[2:].zfill(2)}{hex(rgb[1]).upper()[2:].zfill(2)}{hex(rgb[2]).upper()[2:].zfill(2)}')
print('=' * 58)